<a href="https://colab.research.google.com/github/adnannurrohman8/Project-PSD-Kelompok-J/blob/anggota2-feature%26algo1/NOTEBOOK/Project_PSD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Penelaahan Data**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("ruchikakumbhar/placement-prediction-dataset")

print("Path to dataset files:", path)
data = pd.read_csv(os.path.join(path, "placementdata.csv"))
data.head()
#menampilkan dimensi dataset
data.shape

data.info()
#Mengidentifikasi variabel numerik dan kategorik
numerik = data.select_dtypes(include=['int64', 'float64'])
numerik = numerik.drop(columns=['StudentID'])
kategorik = data.select_dtypes(include=['object'])

print("Variabel Numerik:")
print(numerik.columns.tolist())

print("\nVariabel Kategorik:")
print(kategorik.columns.tolist())

print("\nVariabel Identitas:")
print(['StudentID'])

In [ ]:
# Menghitung statistik deskriptif variabel numerik
numerik = data.select_dtypes(include=['int64', 'float64'])

# Mengeluarkan StudentID
numerik = numerik.drop(columns=['StudentID'])

numerik.describe()

In [ ]:
data.describe(include='object')
#Memeriksa distribusi kelas target
data['PlacementStatus'].value_counts()

data['PlacementStatus'].value_counts(normalize=True) * 100

In [ ]:
#visualisasi distribbusi target
plt.figure(figsize=(7,5))

sns.countplot(data=data, x='PlacementStatus')

plt.title('Distribusi Status Penempatan Kerja')
plt.xlabel('Placement Status')
plt.ylabel('Jumlah Mahasiswa')

plt.show()

#visualisasi distribusi  variabel numerik
data.select_dtypes(include=['int64', 'float64']).drop(
    columns=['StudentID']
).hist(figsize=(15,10), bins=15)

plt.suptitle('Distribusi Variabel Numerik')
plt.tight_layout()
plt.show()

#visualisasi boxplot
plt.figure(figsize=(12,6))

sns.boxplot(data=numerik)

plt.title('Boxplot Variabel Numerik')
plt.xticks(rotation=45)

plt.show()

#visualisasi heatmap korelasi
plt.figure(figsize=(12,8))

sns.heatmap(
    data.select_dtypes(include=['int64', 'float64']).drop(
        columns=['StudentID']
    ).corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Heatmap Korelasi Antarvariabel Numerik')

plt.show()

**2. Validasi Data**

In [ ]:
#Validasi kualitas data
#Memeriksa missing value
data.isnull().sum()

In [ ]:
#Memeriksa data duplikat
data.duplicated().sum()

In [ ]:
#Memeriksa duplikasi ID
data['StudentID'].duplicated().sum()

In [ ]:
#Memeriksa konsistensi kategori
for kolom in ['ExtracurricularActivities',
              'PlacementTraining',
              'PlacementStatus']:

    print(kolom)
    print(data[kolom].value_counts(dropna=False))
    print()

In [ ]:
#Memeriksa rentang nilai numerik
data.describe().T[['min', 'max']]

In [ ]:
#Memeriksa indikasi pencilan menggunakan IQR
# 'numerik' already excludes 'StudentID' from the previous step
for kolom in numerik.columns:

    Q1 = data[kolom].quantile(0.25)
    Q3 = data[kolom].quantile(0.75)

    IQR = Q3 - Q1

    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    pencilan = data[
        (data[kolom] < batas_bawah) |
        (data[kolom] > batas_atas)
    ]

    print(kolom, ":", len(pencilan), "indikasi pencilan")

In [ ]:
#Pembersihan data
data_clean = data.copy()

# Menghapus data duplikat
data_clean = data_clean.drop_duplicates()

# Memperbaiki tipe data kategorik
kolom_kategorik = [
    'ExtracurricularActivities',
    'PlacementTraining',
    'PlacementStatus'
]

for kolom in kolom_kategorik:
    data_clean[kolom] = data_clean[kolom].astype(str).str.strip()

# Memastikan jumlah data setelah pembersihan
print("Dimensi data awal:", data.shape)
print("Dimensi data setelah pembersihan:", data_clean.shape)

# Memeriksa kembali missing value
print("\nMissing Value:")
print(data_clean.isnull().sum())

# Memeriksa kembali duplikat
print("\nJumlah Duplikat:")
print(data_clean.duplicated().sum())

**3. Penentuan Label Data**

In [ ]:
# Menggunakan data hasil pembersihan dari tahap sebelumnya
df = data_clean.copy()

# 1) Pemeriksaan target: tidak ada nilai hilang dan tepat dua kelas
print("Missing value pada target :", df['PlacementStatus'].isnull().sum())
print("Kelas pada target         :", sorted(df['PlacementStatus'].unique()))
assert df['PlacementStatus'].isnull().sum() == 0, "Target memiliki nilai hilang"
assert df['PlacementStatus'].nunique() == 2, "Target harus memiliki tepat dua kelas"

# 2) Pengkodean label: Placed = 1 (kelas positif), NotPlaced = 0 (kelas negatif)
pemetaan_label = {'NotPlaced': 0, 'Placed': 1}
assert set(df['PlacementStatus'].unique()) == set(pemetaan_label.keys()), \
    "Ada nilai target di luar Placed/NotPlaced (cek penulisan kategori)"

y = df['PlacementStatus'].map(pemetaan_label)

# 3) Pemeriksaan konsistensi hasil pengkodean
print("\nDistribusi label (jumlah):")
print(y.value_counts().sort_index())
print("\nDistribusi label (persen):")
print((y.value_counts(normalize=True).sort_index() * 100).round(2))
print("\nMissing value setelah pengkodean:", y.isnull().sum())

**4. Penentuan Objek Data**

In [ ]:
# Daftar fitur yang digunakan
fitur_numerik = ['CGPA', 'Internships', 'Projects', 'Workshops/Certifications',
                 'AptitudeTestScore', 'SoftSkillsRating', 'SSC_Marks', 'HSC_Marks']
fitur_kategorik = ['ExtracurricularActivities', 'PlacementTraining']

# Variabel yang dikeluarkan
kolom_dikeluarkan = ['StudentID', 'PlacementStatus']

# Validasi: semua kolom ada, target tidak menjadi fitur, kolom identitas dikeluarkan
semua_fitur = fitur_numerik + fitur_kategorik
assert all(k in df.columns for k in semua_fitur), "Ada fitur yang tidak ditemukan di dataset"
assert 'PlacementStatus' not in semua_fitur, "Target tidak boleh menjadi fitur"
assert 'StudentID' not in semua_fitur, "Kolom identitas tidak boleh menjadi fitur"

# Pastikan tidak ada kolom lain yang terlewat
sisa = set(df.columns) - set(semua_fitur) - set(kolom_dikeluarkan)
print("Kolom yang belum diklasifikasikan:", sisa if sisa else "tidak ada")

print("\nJumlah fitur numerik   :", len(fitur_numerik))
print("Jumlah fitur kategorik :", len(fitur_kategorik))
print("Total fitur            :", len(semua_fitur))

In [ ]:
# Pemeriksaan awal indikasi data leakage:
# fitur dengan korelasi sangat tinggi terhadap target patut dicurigai
cek = df[fitur_numerik].copy()
for k in fitur_kategorik:
    cek[k] = (df[k] == 'Yes').astype(int)

korelasi_target = cek.corrwith(y).sort_values(ascending=False)
print(korelasi_target.round(3))

mencurigakan = korelasi_target[korelasi_target.abs() > 0.9]
print("\nFitur dengan |korelasi| > 0.9 terhadap target:",
      mencurigakan.index.tolist() if len(mencurigakan) else "tidak ada")

**5. Konstruksi Data**

In [ ]:
# 1) Pembentukan fitur baru
df['TotalExperience'] = df['Internships'] + df['Projects'] + df['Workshops/Certifications']
fitur_numerik_final = fitur_numerik + ['TotalExperience']

X = df[fitur_numerik_final + fitur_kategorik]
print("Dimensi X:", X.shape, "| Dimensi y:", y.shape)
X.head()

In [ ]:
from sklearn.model_selection import train_test_split

# 2) Pembagian data latih dan uji 80:20 (stratified, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)

# Pemeriksaan proporsi kelas (harus mirip pada data awal, latih, dan uji)
proporsi = pd.DataFrame({
    'Data awal (%)': y.value_counts(normalize=True).sort_index() * 100,
    'Data latih (%)': y_train.value_counts(normalize=True).sort_index() * 100,
    'Data uji (%)': y_test.value_counts(normalize=True).sort_index() * 100
}).round(2)
proporsi.index = ['NotPlaced (0)', 'Placed (1)']
proporsi

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 3) Transformasi: imputasi + scaling (numerik) dan imputasi + encoding (kategorik)
pipeline_numerik = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

pipeline_kategorik = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='if_binary', handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', pipeline_numerik, fitur_numerik_final),
    ('cat', pipeline_kategorik, fitur_kategorik)
], verbose_feature_names_out=False)
preprocessor.set_output(transform='pandas')

# 4) fit hanya pada data latih, lalu terapkan ke data uji
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print("Dimensi X_train_prep:", X_train_prep.shape)
print("Dimensi X_test_prep :", X_test_prep.shape)
print("Kolom hasil transformasi:", X_train_prep.columns.tolist())
X_train_prep.head()

In [ ]:
# Verifikasi hasil scaling: parameter dipelajari dari data latih saja
print("Data LATIH setelah scaling (rata-rata harus ~0, std ~1):")
print(X_train_prep[fitur_numerik_final].agg(['mean', 'std']).round(3))

print("\nData UJI setelah scaling (tidak harus tepat 0 dan 1, karena memakai parameter data latih):")
print(X_test_prep[fitur_numerik_final].agg(['mean', 'std']).round(3))

print("\nMissing value X_train_prep:", int(X_train_prep.isnull().sum().sum()))
print("Missing value X_test_prep :", int(X_test_prep.isnull().sum().sum()))

In [ ]:
# Menyimpan data hasil split dan transformasi agar identik dipakai kedua model
# (X_train_prep, X_test_prep, y_train, y_test dipakai oleh Anggota 3 dan 4)
os.makedirs('split_data', exist_ok=True)
X_train_prep.to_csv('split_data/X_train_prep.csv', index=False)
X_test_prep.to_csv('split_data/X_test_prep.csv', index=False)
y_train.to_csv('split_data/y_train.csv', index=False)
y_test.to_csv('split_data/y_test.csv', index=False)
print("Data split tersimpan di folder split_data/")

**6. Pemodelan Algoritma 1: Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Membangun dan melatih model Random Forest
model_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
model_rf.fit(X_train_prep, y_train)

# Prediksi pada data latih dan data uji (disimpan untuk tahap evaluasi oleh Anggota 4)
y_pred_train_model1 = model_rf.predict(X_train_prep)
y_pred_model1 = model_rf.predict(X_test_prep)
y_proba_model1 = model_rf.predict_proba(X_test_prep)[:, 1]  # probabilitas kelas Placed (1)

print("Model Random Forest berhasil dilatih.")
print("Jumlah pohon :", model_rf.n_estimators)
print("Jumlah fitur :", model_rf.n_features_in_)

In [ ]:
# Menampilkan hasil prediksi Random Forest pada data uji
label_balik = {0: 'NotPlaced', 1: 'Placed'}

hasil_prediksi = pd.DataFrame({
    'Aktual': y_test.map(label_balik).values,
    'Prediksi': pd.Series(y_pred_model1).map(label_balik).values,
    'Probabilitas Placed': y_proba_model1.round(3)
}, index=y_test.index)

print("10 prediksi pertama pada data uji:")
display(hasil_prediksi.head(10))

print("\nDistribusi hasil prediksi pada data uji:")
print(hasil_prediksi['Prediksi'].value_counts())

In [ ]:
# Feature importance Random Forest
importance = pd.Series(model_rf.feature_importances_, index=X_train_prep.columns).sort_values()

plt.figure(figsize=(8, 5))
importance.plot(kind='barh')
plt.title('Feature Importance Random Forest')
plt.xlabel('Tingkat Kepentingan')
plt.tight_layout()
plt.show()

importance.sort_values(ascending=False).round(4)